# Take a selection into a fresh analysis

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/r_package/f_notebooks_tutorials/12_subset_and_reanalyse.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Select a population in the viewer and re-analyse just those cells:
from a live viewer with `cellucid_subset()`, or from the web app's
exports with no R session attached.

**Time:** 15 minutes. **You need:** `Seurat`. The same tutorial in
Python: [Export an .h5ad subset and re-analyse](../../python_package/f_notebooks_tutorials/46_export_an_h5ad_subset_and_reanalyse.ipynb).

In [ ]:
for (package in c("remotes", "SeuratObject", "Seurat")) {
  if (!requireNamespace(package, quietly = TRUE)) install.packages(package)
}
if (!requireNamespace("cellucid", quietly = TRUE)) {
  if ("cellucid" %in% rownames(available.packages())) install.packages("cellucid") else remotes::install_github("theislab/cellucid-r")
}

## The data

SeuratObject's own example, `pbmc_small`: 80 blood cells, 230 genes, a
t-SNE and the clusters `RNA_snn_res.1`. Use your own Seurat or
SingleCellExperiment object in its place.

In [ ]:
library(cellucid)
library(SeuratObject)
data("pbmc_small")
seu <- pbmc_small
seu

## From a live viewer

Lasso a population (colour by `groups` or `RNA_snn_res.1` to find
one) and click **Confirm**, then run the next cell.

In [ ]:
viewer <- cellucid_view(seu, dataset_id = "pbmc-small", dataset_name = "PBMC small")

In [ ]:
picked <- cellucid_subset(seu, viewer)  # which = NULL: the last selection
picked

## Without a live viewer: the CSV export

In the web app, highlight the cells and click **CSV** beside
**Clear**: the file lists the highlighted cells by name. Read it and
subset by name (or `cellucid_pull()` it, then subset by the column):

In [ ]:
csv <- "Page_1_cells.csv"
if (file.exists(csv)) {
  names_in_file <- read.csv(csv, check.names = FALSE)$obs_name
  picked <- subset(seu, cells = names_in_file)
}
picked

The web app's **.h5ad** export (the highlighted cells with their
fields, the expression rows of the matrix the viewer reads, and the
embeddings) is an ordinary AnnData file: Bioconductor's `anndataR`
reads it into a SingleCellExperiment (or, with `as = "Seurat"`, a
Seurat object).

In [ ]:
h5ad <- "Page_1_cells.h5ad"
if (file.exists(h5ad) && requireNamespace("anndataR", quietly = TRUE)) {
  exported <- anndataR::read_h5ad(h5ad, as = "SingleCellExperiment")
  print(exported)
}

## Re-analyse

In [ ]:
library(Seurat)
picked <- NormalizeData(picked, verbose = FALSE)
picked <- FindVariableFeatures(picked, nfeatures = 100, verbose = FALSE)
picked <- ScaleData(picked, verbose = FALSE)
picked <- RunPCA(picked, npcs = 5, verbose = FALSE)
picked <- FindNeighbors(picked, dims = 1:5, k.param = 5, verbose = FALSE)
picked <- FindClusters(picked, resolution = 0.8, verbose = FALSE)
table(picked$seurat_clusters)

The subset is an object like any other, so the same line shows it:

In [ ]:
sub_viewer <- cellucid_view(picked, dataset_id = "pbmc-small-subset", dataset_name = "PBMC small: subset")

In [ ]:
cellucid_stop(sub_viewer)
cellucid_stop(viewer)

## Next

- Python: [Export an .h5ad subset and re-analyse](../../python_package/f_notebooks_tutorials/46_export_an_h5ad_subset_and_reanalyse.ipynb)